In [ ]:
import pandas as pd
import os
import numpy as np

df = pd.read_csv(r'E:\University of Michigan Dropbox\Jiahui Chen\PVEV\PVcompensation\ratebooks\CA_southern california edison co_TOU-D-PRIM.csv')

In [ ]:
# Convert DateStart to datetime for easier processing
df['DateStart'] = pd.to_datetime(df['DateStart'])

# Filter groups with exactly 8752 rows
time_series_data = {}
for (year, rin), group in df.groupby([df['DateStart'].dt.year, 'RIN']):
    time_series_data[(year, rin)] = group.sort_values(['DateStart', 'TimeStart'])

# Display the keys of the extracted time series data
print(f"Extracted time series for {len(time_series_data)} unique combinations.")

In [ ]:
credit = time_series_data[2025,'USCA-SCXX-NB25-0000']['Value'].to_numpy() + time_series_data[2025,'USCA-XXSC-NB25-0000']['Value'].to_numpy()
credit1 = np.append(np.zeros(8), credit)
result_df = pd.DataFrame({
    'Rate (cent/kWh)': credit1*100,
    'fixed_monthly_charge': 0
})
result_df.to_csv(r'E:\University of Michigan Dropbox\Jiahui Chen\PVEV\PVcompensation\rates\CA_southern california edison co_TOU-D-PRIM.csv')

In [ ]:
df = pd.read_excel(r'E:\University of Michigan Dropbox\Jiahui Chen\PVEV\PVcompensation\rates\CA_san diego gas & electric co_EV-TOU-2.xlsx')
df['Rate (cent/kWh)'] = df['Rate (cent/kWh)'] *100
df.to_csv(r'E:\University of Michigan Dropbox\Jiahui Chen\PVEV\PVcompensation\rates\CA_san diego gas & electric co_EV-TOU-2.csv', index=False)

In [ ]:
#pacific gas and electric co
df = pd.read_excel(r'E:\University of Michigan Dropbox\Jiahui Chen\PVEV\PVcompensation\ratebooks\CA_pacific gas & electric co_Electric Vehicle EV2 (Sch).xlsx', header=None)
df.to_csv(r'E:\University of Michigan Dropbox\Jiahui Chen\PVEV\PVcompensation\rates\CA_pacific gas & electric co_Electric Vehicle EV2 (Sch).csv', index=False)


In [ ]:
df = pd.read_csv(r'E:\University of Michigan Dropbox\Jiahui Chen\PVEV\PVcompensation\rates\CA_pacific gas & electric co_Electric Vehicle EV2 (Sch).csv', header=None, sep=r',|\s\$', engine='python')
# df = pd.read_csv(r'E:\University of Michigan Dropbox\Jiahui Chen\PVEV\PVcompensation\rates\CA_pacific gas & electric co_Electric Vehicle EV2 (Sch).csv')
# Calculate monthly rates by summing the respective columns
monthly_rates = pd.DataFrame({
    'January': df.iloc[:, [2, 3]].sum(axis=1),
    'February': df.iloc[:, [4, 5]].sum(axis=1),
    'March': df.iloc[:, [6, 7]].sum(axis=1),
    'April': df.iloc[:, [8, 9]].sum(axis=1),
    'May': df.iloc[:, [10, 11]].sum(axis=1),
    'June': df.iloc[:, [12, 13]].sum(axis=1),
    'July': df.iloc[:, [14, 15]].sum(axis=1),
    'August': df.iloc[:, [16, 17]].sum(axis=1),
    'September': df.iloc[:, [18, 19]].sum(axis=1),
    'October': df.iloc[:, [20, 21]].sum(axis=1),
    'November': df.iloc[:, [22, 23]].sum(axis=1),
    'December': df.iloc[:, [24, 25]].sum(axis=1)
})
weekday_rates = monthly_rates.iloc[:24].reset_index(drop=True)
weekend_holiday_rates = monthly_rates.iloc[24:].reset_index(drop=True)

weekday_rates, weekend_holiday_rates

In [ ]:
from pandas.tseries.holiday import USFederalHolidayCalendar

# Generate 8760-hour time series data for 9 years starting from 2025
for start_year in range(2025,2025+9):
    end_year = start_year + 1  # 9 years including 2025
    time_index = pd.date_range(start=f'{start_year}-01-01 00:00:00', end=f'{end_year}-01-01 00:00:00', freq='H')

    # Create a DataFrame to hold the time series data
    time_series_df = pd.DataFrame({'Datetime': time_index})
    time_series_df['Year'] = time_series_df['Datetime'].dt.year
    time_series_df['Month'] = time_series_df['Datetime'].dt.month
    time_series_df['Day'] = time_series_df['Datetime'].dt.day
    time_series_df['Hour'] = time_series_df['Datetime'].dt.hour
    time_series_df['Weekday'] = time_series_df['Datetime'].dt.weekday

    # Identify US federal holidays
    cal = USFederalHolidayCalendar()
    holidays = cal.holidays(start=f'{start_year}-01-01', end=f'{end_year}-12-31')
    time_series_df['IsHoliday'] = time_series_df['Datetime'].dt.date.isin(holidays.date)

    # Assign rates based on weekday/weekend/holiday
    rates = []
    for _, row in time_series_df.iterrows():
        if row['IsHoliday'] or row['Weekday'] >= 5:  # Weekend or holiday
            rates.append(weekend_holiday_rates.iloc[row['Hour'], row['Month'] - 1]*100)
        else:  # Weekday
            rates.append(weekday_rates.iloc[row['Hour'], row['Month'] - 1]*100)

    time_series_df['Rate (cent/kWh)'] = rates
    time_series_df['fixed_monthly_charge'] = 0
    time_series_df = time_series_df[['Rate (cent/kWh)', 'fixed_monthly_charge']]
    # Display the first few rows of the generated time series data
    print(time_series_df.head())
    print(time_series_df.tail())
    time_series_df.to_csv(fr'E:\University of Michigan Dropbox\Jiahui Chen\PVEV\PVcompensation\rates\CA_pacific gas & electric co_Electric Vehicle EV2 (Sch)_{start_year}.csv')